### モジュールのインポート

In [ ]:
import os
import glob
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import datetime
#from tqdm import tqdm
from tqdm.notebook import tqdm
import pickle
import random
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from PIL import Image
import skimage.transform
from collections import deque
from typing import Sequence, Dict, Tuple, Union

import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence
from torchvision import models
import torchvision.transforms as T
import torchvision.datasets as dataset
from torchvision.transforms import v2

from timm.scheduler import CosineLRScheduler
from transformers import  get_linear_schedule_with_warmup

#from transformers import AutoImageProcessor, AutoModel, AutoProcessor, CLIPVisionModel
from transformers import BertTokenizer, BertModel, CLIPVisionModel, BertForPreTraining

import sys

import util
import levenshtein
from nltk import bleu_score
import ssl
from torch.amp import autocast, GradScaler

### 位置エンコーディングの実装

In [ ]:
class PositionalEmbedding(nn.Module):
    '''
    位置埋め込み （Positional embedding）
    dim_embedding: 埋込み次元
    max_len      : 入力の最大系列長
    '''
    def __init__(self, dim_embedding: int, max_len: int=2048):
        super().__init__()

        self.pos_emb = nn.Embedding(max_len, dim_embedding)

    '''
    位置エンコーディングの順伝播
    x: 位置エンコーディングを埋め込む対象のテンソル,
       [バッチサイズ, 系列長, 埋め込み次元]
    '''
    def forward(self, x: torch.Tensor):
        seq = x.shape[1]
        positions = torch.arange(start=0, end=seq, step=1, device=x.device).to(torch.long)
        positions = self.pos_emb(positions)[:seq,:]
        
        return positions

### CRF 層の定義

In [ ]:
def logsumexp(x, dim=1):
    return torch.logsumexp(x.float(), dim=dim).type_as(x)

class DynamicCRF(nn.Module):
    def __init__(self, num_embedding, low_rank=32, beam_size=64, crf_coef=1.0, temp = 0.5, top_p = 0.9, top_k = 50, 
                 num_samples = 16, ref_t = False ):
        super().__init__()

        #low_rank = num_embedding
        self.E1 = nn.Embedding(num_embedding, low_rank)
        self.E2 = nn.Embedding(num_embedding, low_rank)

        self.vocb = num_embedding
        self.rank = low_rank
        self.beam = beam_size
        self.crf_coef = crf_coef
        self.temp = temp
        self.top_p = top_p
        self.top_k = top_k
        self.num_samples = num_samples
        self.ref_t = ref_t
        
    def extra_repr(self):
        return "vocab_size={}, low_rank={}, beam_size={}".format(
            self.vocb, self.rank, self.beam)

    def forward(self, emissions, top_logits, top_indices, targets, masks, beam=None):
        numerator = self._compute_score(emissions, targets, masks)
        denominator = self._compute_normalizer(emissions, targets, masks, beam )

        return numerator - denominator
    
    def forward_decoder(self, emissions, masks=None, beam=None):
        return self._viterbi_decode(emissions, masks, beam)

    def _compute_score(self, emissions, targets, masks=None):
        batch_size, seq_len = targets.size()
        emission_scores = emissions.gather(2, targets[:, :, None])[:, :, 0]  # B x T
        transition_scores = (self.E1(targets[:, :-1]) * self.E2(targets[:, 1:])).sum(2)
        
        scores = emission_scores
        scores[:, 1:] += transition_scores
        
        #if masks is not None:
        #    scores = scores * masks.type_as(scores)

        return scores.sum(-1)
    
    def _compute_normalizer(self, emissions, targets=None, masks=None, beam=None):

        eps = 1e-8
        
        beam = beam if beam is not None else self.beam
        batch_size, seq_len = emissions.size()[:2]

        if targets is not None:
            #_emissions = emissions.scatter(2, targets[:, :, None], np.float('inf'))
            _emissions = emissions.scatter(2, targets[:, :, None], float('inf'))
            beam_targets = _emissions.topk(beam, 2)[1]
            beam_emission_scores = emissions.gather(2, beam_targets)
        else:
            beam_emission_scores, beam_targets = emissions.topk(beam, 2)
        beam_transition_score1 = self.E1(beam_targets[:, :-1])  # B x (T-1) x K x D; position i - 1, previous step.
        beam_transition_score2 = self.E2(beam_targets[:, 1:])   # B x (T-1) x K x D; position i, current step.
        beam_transition_matrix = torch.bmm(
            beam_transition_score1.view(-1, self.beam, self.rank),
            beam_transition_score2.view(-1, self.beam, self.rank).transpose(1, 2))
        beam_transition_matrix = beam_transition_matrix.view( batch_size, -1, self.beam, self.beam)

        # compute the normalizer in the log-space
        score = beam_emission_scores[:, 0]  # B x K
        for i in range(1, seq_len):
            next_score = score[:, :, None] + beam_transition_matrix[:, i-1]
            next_score = logsumexp(next_score, dim=1) + beam_emission_scores[:, i]
            tmp = logsumexp( next_score, dim = 1 )
            if masks is not None:
                score = torch.where(masks[:, i:i+1], next_score, score)
            else:
                score = next_score

        normalizer = logsumexp(score, dim=1)
        return normalizer
    
    def _compute_grpo_samples(self, beam_emission_scores, beam_transition_matrix, beam_targets, \
                              targets=None, masks=None, beam=None):
        
        eps = 1e-8
        device = beam_emission_scores.device

        beam = beam if beam is not None else self.beam
        batch_size, seq_len = beam_emission_scores.size()[:2]

        # フィルタリング用のパラメータ設定 (config等から取得できるよう適宜調整してください)
        #top_k = 50  # 上位k個に絞る (0なら無効)
        #top_p = 0.9 # 累積確率pまでに絞る (1.0なら無効)


        traj_tokens, traj_scores = [], []
        traj_scores2 = []
        traj_tokens2 = []
        #selected_log_probs = []
        selected_probs = []
        #entropy = []

        score = beam_emission_scores[:, 0]
        _score2 = beam_emission_scores[:,0][:,None,:].expand( -1, self.num_samples, -1 )
        _score3 = beam_emission_scores[:,0][:,None,:,None].expand( -1, self.num_samples, -1, beam )
        B, N, C, W = _score3.shape
        flat_score = _score3.permute(0, 3, 1, 2).reshape(-1, C)
        logits = flat_score / self.temp # B*W*N, C 
        probs = F.softmax(logits, dim=-1)  # B*W*N,C この softmax は、C についての softmax          
        # 安全策: 全てが -Inf になった場合の回避
        if torch.isnan(probs).any():
            probs = torch.ones_like(probs) / probs.size(-1)
        #torch.manual_seed(42)
        _index_flat = torch.multinomial(probs, num_samples=1, replacement=True)  
        #_index_flat = torch.argmax( probs, dim = -1 ).unsqueeze( -1 )
        
        all_inf_mask = (logits == float('-inf')).all(dim=-1)
        if all_inf_mask.any():
            # 全て-infの行だけ、一様な値をいれる
            logits[all_inf_mask] = 0.0 

        #log_probs_full = F.log_softmax(logits, dim=-1)
        #probs_full = torch.exp(log_probs_full)
        #selected_log_prob = torch.gather(log_probs_full, -1, _index_flat) # (B*W*N, 1)
        #selected_log_probs.append( selected_log_prob.view( B, W, self.num_samples ).transpose(1,2) )
        #selected_prob = torch.gather( probs_full, -1, _index_flat)
        #selected_probs.append( selected_prob.view( B, W, self.num_samples).transpose( 1, 2 ) )
        #step_entropy = - (probs_full * log_probs_full).sum(dim=-1) # (B*W*N,) 
        #step_entropy = step_entropy.view( B, W, N )
        #entropy.append( step_entropy )
        #probs_full = F.softmax( logits, dim = -1 )
        selected_prob = torch.gather( probs, -1, _index_flat)
        selected_probs.append( selected_prob.view( B, W, self.num_samples) )  # S, B, N, beam
        
        for i in range(1, seq_len):
            traj_scores.append(score)
            traj_scores2.append( _score2 )
            _score2 = _score2[:,:,:,None] + beam_transition_matrix[:, i-1,None,:,:].expand( -1, self.num_samples,-1,-1) 

            B, N, C, W = _score2.shape
            flat_score = _score2.permute(0, 3, 1, 2).reshape(-1, C)
            
            # --- Top-K / Top-P Filtering 開始 (修正版) ---
            logits = flat_score / self.temp #B,W,N,C
            
            # 1. まず Top-K で上位K個に絞る (0なら無効)
            if self.top_k > 0:
                top_k_val = min(self.top_k, logits.size(-1)) # top_kが語彙数より大きくならないように調整
                top_k_logits, top_k_indices = torch.topk(logits, top_k_val, dim=-1)
                
                # Top-K用のマスクを作成
                min_values = top_k_logits[:, -1].unsqueeze(-1)
                logits = torch.where(logits < min_values, torch.full_like(logits, float('-inf')), logits)
                
                # 以降の処理（Top-P）のために top_k_logits, top_k_indices を更新
                # （注: top_pと組み合わせる場合、ここでのlogitsの更新より、
                #  後続のtop_k_indicesを使ったmaskの方がロジックが整合しやすい）
            
            # Top-Kの変数を再定義（top_pの処理で使うため）
            # top_k > 0 の場合、top_kで絞った後の値を使う。0の場合は全範囲。
            top_k_logits, top_k_indices = logits, torch.arange(logits.size(-1), device=logits.device).expand(logits.size(0), -1) 
            # ↑ このアプローチはメモリを食うため、元の実装の通りtop_k_indicesでmaskする方が綺麗です。
            # 以下、元のロジックを活かした修正版です。
            # 2. Top-P (Nucleus) filtering
            if self.top_p < 1.0:
                # Top-K/Allで絞ったテンソルでソート
                sorted_logits, sorted_indices = torch.sort(top_k_logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                
                # 除去対象のマスクを作成
                sorted_indices_to_remove = cumulative_probs > self.top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                
                # top_k_logitsと同じ形状のマスクを作成
                indices_to_remove_k = torch.zeros_like(top_k_logits, dtype=torch.bool).scatter_(
                    dim=-1, index=sorted_indices, src=sorted_indices_to_remove
                )
                top_k_logits[indices_to_remove_k] = -float('Inf')
    
            # 3. 全語彙の logits を一旦すべて -inf にして、生き残った top_k_logits だけを戻す
            new_logits = torch.full_like(logits, float('-inf'))
            new_logits.scatter_(dim=-1, index=top_k_indices, src=top_k_logits)
            logits = new_logits
            
            probs = F.softmax(logits, dim=-1) # B*W*N,C この softmax は、C についての softmax
           
            # 安全策: 全てが -Inf になった場合の回避
            if torch.isnan(probs).any():
                probs = torch.ones_like(probs) / probs.size(-1)

            #torch.manual_seed(42) 
            _index_flat = torch.multinomial(probs, num_samples=1, replacement=True) 
            #_index_flat = torch.argmax( probs, dim = -1 ).unsqueeze( -1 )
            
            all_inf_mask = (logits == float('-inf')).all(dim=-1)
            if all_inf_mask.any():
                # 全て-infの行だけ、一様な値をいれる
                logits[all_inf_mask] = 0.0 

            #log_probs_full = F.log_softmax(logits, dim=-1) #B,W,N
            #probs_full = torch.exp(log_probs_full)
            #selected_log_prob = torch.gather(log_probs_full, -1, _index_flat) # (B*W*N, 1)
            #selected_log_probs.append( selected_log_prob.view( B, W, self.num_samples ).transpose(1,2) )
            #selected_prob = torch.gather( probs_full, -1, _index_flat)
            #selected_probs.append( selected_prob.view( B, W, self.num_samples).transpose( 1, 2 ) )
            ##step_entropy = - (probs_full * log_probs_full).sum(dim=-1) # (B*W*N,) 
            ##step_entropy = step_entropy.view( B, W, N )
            ##entropy.append( step_entropy ) #S, B, W, N
            #probs_full = F.softmax( logits, dim = -1 ) 
            selected_prob = torch.gather( probs, -1, _index_flat)
            selected_probs.append( selected_prob.view( B, W, self.num_samples) ) # S, B, W = beam, N
            
            _score_flat = torch.gather(flat_score, -1, _index_flat)
            _index2 = _index_flat.view(B, W, self.num_samples).transpose(1,2)
            _score2 = _score_flat.view(B, W, self.num_samples).transpose(1,2)

            _score2 = _score2 + beam_emission_scores[:, i][:,None,:].expand(-1,self.num_samples,-1)

            score, index = _score2[:,0,:], _index2[:,0,:]
            traj_tokens.append(index)
            traj_tokens2.append( _index2 )

        #selected_log_probs = torch.stack( selected_log_probs, dim = 0 )
        #log_probs = selected_log_probs.permute( 1, 0, 3, 2 )
        selected_probs = torch.stack( selected_probs, dim = 0 ) # S, B, beam, N
        beam_probs = selected_probs.transpose( 0, 1 ) # B, S, beam, N
        beam_probs = F.softmax( beam_probs, dim = 2 ) # B, S, beam, N → B, S, N   W についての softmax 一つ一つの B,S,N について、W = beam についての総和が1の確率
        log_probs = torch.log( beam_probs )
        #beam_probs = torch.exp( log_probs )
        #entropy = torch.stack( entropy, dim = 0 ) #S,B,W,N
        #entropy = torch.mean( entropy, dim = 2 )  #S,B, N
        #entropy = entropy.permute( 1, 0, 2 ) #B,S,N

        # --- 以下、元のバックトレーシング処理 ---
        #all_scores = traj_scores2
        #all_scores.append( _score2 )
        #all_scores = torch.stack( all_scores, dim = 0 ).transpose( 0, 1 ).to(device)
        #beam_probs = F.softmax( all_scores.transpose( 2, 3 ), dim = 2 )

        # --- 修正：バックトレーシングの開始 ---
        # 1. まずリストを空にする（重要！）
        finalized_tokens = []
        #finalized_scores = []
        
        ## 2. サンプリングされた最後のインデックスを取得 (B, N, 1)
        B, N, C = _score2.shape
        flat_score = _score2.permute(0, 1, 2).reshape(-1, C)

        # 1. 温度パラメータが0になっていないかチェック (0の場合は直接 argmax を使うなどの処理が必要)
        temp = max(self.temp, 1e-6) 

        # 2. スコアの計算
        scaled_score = flat_score / temp

        # 3. もし scaled_score に NaN や inf があれば 0 や大きな値に置換して防御
        if torch.isnan(scaled_score).any() or torch.isinf(scaled_score).any():
            # NaN は 0 に、inf は有限の大きな値に置き換える
            scaled_score = torch.nan_to_num(scaled_score, nan=0.0, posinf=20.0, neginf=-20.0)

        # 4. Softmax の計算
        probs = F.softmax(scaled_score, dim=-1)

        ## 5. 確率の微小な誤差で NaN になるのを防ぐ安全処理
        #probs = torch.nan_to_num(probs, nan=0.0, posinf=1.0, neginf=0.0)
        #probs = probs / (probs.sum(dim=-1, keepdim=True) + 1e-9) # 再正規化

        # 5. アンダーフロー（すべて0になる現象）を防ぐための安全弁
        # 万が一、すべての確率が 0 になった場合は均等な確率にする
        zero_mask = (probs.sum(dim=-1, keepdim=True) <= 0)
        if zero_mask.any():
            # 確率が0の行を、均等な確率（1 / 要素数）で埋める
            uniform_probs = torch.ones_like(probs) / probs.size(-1)
            probs = torch.where(zero_mask, uniform_probs, probs)

        # 6. 再正規化（微小値を足してから割る）
        probs = probs + 1e-9
        probs = probs / probs.sum(dim=-1, keepdim=True)
        
        #probs = F.softmax(flat_score / self.temp, dim=-1)
        #torch.manual_seed(42) 
        _index_flat = torch.multinomial(probs, num_samples=1, replacement = True )  
        #_index_flat = torch.argmax( probs, dim = -1 ).unsqueeze( -1 )
        
        _score_flat = torch.gather(flat_score, -1, _index_flat)
        _index2 = _index_flat.view(B, self.num_samples, 1)
        _score2 = _score_flat.view(B, self.num_samples, 1)
        current_sampled_index = _index2[:, :] #(B, N, 1 )

        ## 3. 最初の要素として追加
        finalized_tokens.append(current_sampled_index) # (B, N, 1)
        #finalized_scores.append(probs.view( B, N, C ).gather(2, current_sampled_index)) # (B, N, 1)

        # 4. 過去に遡ってパスを復元
        # traj_tokens2 はループ内で append された各ステップのサンプリング結果
        for idx_step, scs_step in zip(reversed(traj_tokens2), reversed(traj_scores2)):
            # 直前のステップで選ばれたインデックスを使って、その前のインデックスを引く
            previous_pointer = finalized_tokens[-1]
            finalized_tokens.append(idx_step.gather(2, previous_pointer))
            #finalized_scores.append(scs_step.gather(2, previous_pointer))

        # 5. 逆順にして結合 (この時点では [seq_len, B, Group, 1] のリスト)
        finalized_tokens.reverse()

        sampled_beam_idx = torch.cat(finalized_tokens, 2) # [B, G, S]
        
        # 2. beam_targets [B, S, Beam] と合わせるために transpose
        # [B, G, S] -> [B, S, G] にして、各ステップ(S)ごとのビーム(G)を選択
        sampled_beam_idx = sampled_beam_idx.permute(0, 2, 1) # [B, S, G]

        # 3. gather 実行 (第2次元[Beam方向]から、サンプリングしたインデックスを抽出)
        finalized_tokens = beam_targets.gather(2, sampled_beam_idx) # [B, S, G]
        
        #finalized_scores.reverse()
        #finalized_scores = torch.cat(finalized_scores, 1)
        #finalized_scores[:, 1:] = finalized_scores[:, 1:] - finalized_scores[:, :-1]
       
        #return log_probs, entropy, sampled_beam_idx, finalized_tokens 
        return log_probs, beam_probs, sampled_beam_idx, finalized_tokens 
    '''
    def _compute_grpo_samples(self, beam_emission_scores, beam_transition_matrix, beam_targets, \
                              targets=None, masks=None, beam=None):
        
        eps = 1e-8
        device = beam_emission_scores.device

        beam = beam if beam is not None else self.beam
        batch_size, seq_len = beam_emission_scores.size()[:2]

        # フィルタリング用のパラメータ設定 (config等から取得できるよう適宜調整してください)
        #top_k = 50  # 上位k個に絞る (0なら無効)
        #top_p = 0.9 # 累積確率pまでに絞る (1.0なら無効)


        traj_tokens, traj_scores = [], []
        traj_scores2 = []
        traj_tokens2 = []
        selected_log_probs = []
        entropy = []

        score = beam_emission_scores[:, 0]
        _score2 = beam_emission_scores[:,0][:,None,:].expand( -1, self.num_samples, -1 )
        _score3 = beam_emission_scores[:,0][:,None,:,None].expand( -1, self.num_samples, -1, beam )
        B, N, C, W = _score3.shape
        flat_score = _score3.permute(0, 3, 1, 2).reshape(-1, C)
        logits = flat_score / self.temp
        probs = F.softmax(logits, dim=-1)            
        # 安全策: 全てが -Inf になった場合の回避
        if torch.isnan(probs).any():
            probs = torch.ones_like(probs) / probs.size(-1)
        #torch.manual_seed(42)
        _index_flat = torch.multinomial(probs, num_samples=1, replacement=True)  
        #_index_flat = torch.argmax( probs, dim = -1 ).unsqueeze( -1 )
        
        all_inf_mask = (logits == float('-inf')).all(dim=-1)
        if all_inf_mask.any():
            # 全て-infの行だけ、一様な値をいれる
            logits[all_inf_mask] = 0.0 

        log_probs_full = F.log_softmax(logits, dim=-1)
        probs_full = torch.exp(log_probs_full)
        selected_log_prob = torch.gather(log_probs_full, -1, _index_flat) # (B*W*N, 1)
        selected_log_probs.append( selected_log_prob.view( B, W, self.num_samples ).transpose(1,2) )
        step_entropy = - (probs_full * log_probs_full).sum(dim=-1) # (B*W*N,) 
        step_entropy = step_entropy.view( B, W, N )
        entropy.append( step_entropy )

        for i in range(1, seq_len):
            traj_scores.append(score)
            traj_scores2.append( _score2 )
            _score2 = _score2[:,:,:,None] + beam_transition_matrix[:, i-1,None,:,:].expand( -1, self.num_samples,-1,-1) 

            B, N, C, W = _score2.shape
            flat_score = _score2.permute(0, 3, 1, 2).reshape(-1, C)
            
            # --- Top-K / Top-P Filtering 開始 (修正版) ---
            logits = flat_score / self.temp #B,W,N,C
            
            # 1. まず Top-K で上位K個に絞る (0なら無効)
            if self.top_k > 0:
                top_k_val = min(self.top_k, logits.size(-1)) # top_kが語彙数より大きくならないように調整
                top_k_logits, top_k_indices = torch.topk(logits, top_k_val, dim=-1)
                
                # Top-K用のマスクを作成
                min_values = top_k_logits[:, -1].unsqueeze(-1)
                logits = torch.where(logits < min_values, torch.full_like(logits, float('-inf')), logits)
                
                # 以降の処理（Top-P）のために top_k_logits, top_k_indices を更新
                # （注: top_pと組み合わせる場合、ここでのlogitsの更新より、
                #  後続のtop_k_indicesを使ったmaskの方がロジックが整合しやすい）
            
            # Top-Kの変数を再定義（top_pの処理で使うため）
            # top_k > 0 の場合、top_kで絞った後の値を使う。0の場合は全範囲。
            top_k_logits, top_k_indices = logits, torch.arange(logits.size(-1), device=logits.device).expand(logits.size(0), -1) 
            # ↑ このアプローチはメモリを食うため、元の実装の通りtop_k_indicesでmaskする方が綺麗です。
            # 以下、元のロジックを活かした修正版です。
            # 2. Top-P (Nucleus) filtering
            if self.top_p < 1.0:
                # Top-K/Allで絞ったテンソルでソート
                sorted_logits, sorted_indices = torch.sort(top_k_logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                
                # 除去対象のマスクを作成
                sorted_indices_to_remove = cumulative_probs > self.top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                
                # top_k_logitsと同じ形状のマスクを作成
                indices_to_remove_k = torch.zeros_like(top_k_logits, dtype=torch.bool).scatter_(
                    dim=-1, index=sorted_indices, src=sorted_indices_to_remove
                )
                top_k_logits[indices_to_remove_k] = -float('Inf')
    
            # 3. 全語彙の logits を一旦すべて -inf にして、生き残った top_k_logits だけを戻す
            new_logits = torch.full_like(logits, float('-inf'))
            new_logits.scatter_(dim=-1, index=top_k_indices, src=top_k_logits)
            logits = new_logits
            
            probs = F.softmax(logits, dim=-1) #B,W,N
           
            # 安全策: 全てが -Inf になった場合の回避
            if torch.isnan(probs).any():
                probs = torch.ones_like(probs) / probs.size(-1)

            #torch.manual_seed(42) 
            _index_flat = torch.multinomial(probs, num_samples=1, replacement=True) 
            #_index_flat = torch.argmax( probs, dim = -1 ).unsqueeze( -1 )
            
            all_inf_mask = (logits == float('-inf')).all(dim=-1)
            if all_inf_mask.any():
                # 全て-infの行だけ、一様な値をいれる
                logits[all_inf_mask] = 0.0 

            log_probs_full = F.log_softmax(logits, dim=-1) #B,W,N
            probs_full = torch.exp(log_probs_full)
            selected_log_prob = torch.gather(log_probs_full, -1, _index_flat) # (B*W*N, 1)
            selected_log_probs.append( selected_log_prob.view( B, W, self.num_samples ).transpose(1,2) )
            step_entropy = - (probs_full * log_probs_full).sum(dim=-1) # (B*W*N,) 
            step_entropy = step_entropy.view( B, W, N )
            entropy.append( step_entropy ) #S, B, W, N
            
            _score_flat = torch.gather(flat_score, -1, _index_flat)
            _index2 = _index_flat.view(B, W, self.num_samples).transpose(1,2)
            _score2 = _score_flat.view(B, W, self.num_samples).transpose(1,2)

            _score2 = _score2 + beam_emission_scores[:, i][:,None,:].expand(-1,self.num_samples,-1)

            score, index = _score2[:,0,:], _index2[:,0,:]
            traj_tokens.append(index)
            traj_tokens2.append( _index2 )

        selected_log_probs = torch.stack( selected_log_probs, dim = 0 )
        log_probs = selected_log_probs.permute( 1, 0, 3, 2 )
        #beam_probs = torch.exp( log_probs )
        entropy = torch.stack( entropy, dim = 0 ) #S,B,W,N
        entropy = entropy.mean( dim = 2 )   #S,B, N
        entropy = entropy.permute( 1, 0, 2 ) #B,S,N

        # --- 以下、元のバックトレーシング処理 ---
        #all_scores = traj_scores2
        #all_scores.append( _score2 )
        #all_scores = torch.stack( all_scores, dim = 0 ).transpose( 0, 1 ).to(device)
        #beam_probs = F.softmax( all_scores.transpose( 2, 3 ), dim = 2 )

        # --- 修正：バックトレーシングの開始 ---
        # 1. まずリストを空にする（重要！）
        finalized_tokens = []
        #finalized_scores = []
        
        # 2. サンプリングされた最後のインデックスを取得 (B, N, 1)
        # _index2 が (B, W, self.num_samples) の場合、適切にリシェイプ
        # ※ W=1, self.num_samples=16 のケースが多いです
        current_sampled_index = _index2[:, :, 0:1] # (B, W, 1) 

        ## 3. 最初の要素として追加
        finalized_tokens.append(current_sampled_index) # (B, N, 1)
        #finalized_scores.append(probs.view( B, N, C ).gather(2, current_sampled_index)) # (B, N, 1)

        # 4. 過去に遡ってパスを復元
        # traj_tokens2 はループ内で append された各ステップのサンプリング結果
        for idx_step, scs_step in zip(reversed(traj_tokens2), reversed(traj_scores2)):
            # 直前のステップで選ばれたインデックスを使って、その前のインデックスを引く
            previous_pointer = finalized_tokens[-1]
            finalized_tokens.append(idx_step.gather(2, previous_pointer))
            #finalized_scores.append(scs_step.gather(2, previous_pointer))

        # 5. 逆順にして結合 (この時点では [seq_len, B, Group, 1] のリスト)
        finalized_tokens.reverse()

        sampled_beam_idx = torch.cat(finalized_tokens, 2) # [B, G, S]
        
        # 2. beam_targets [B, S, Beam] と合わせるために transpose
        # [B, G, S] -> [B, S, G] にして、各ステップ(S)ごとのビーム(G)を選択
        sampled_beam_idx = sampled_beam_idx.permute(0, 2, 1) # [B, S, G]

        # 3. gather 実行 (第2次元[Beam方向]から、サンプリングしたインデックスを抽出)
        finalized_tokens = beam_targets.gather(2, sampled_beam_idx) # [B, S, G]
        
        #finalized_scores.reverse()
        #finalized_scores = torch.cat(finalized_scores, 1)
        #finalized_scores[:, 1:] = finalized_scores[:, 1:] - finalized_scores[:, :-1]
       
        return log_probs, entropy, sampled_beam_idx, finalized_tokens 
    '''    
    def _compute_ref_probs( self, beam_emission_scores, beam_transition_matrix, sampled_beam_idx, masks = None, beam = None ):
        
        eps = 1e-8
        device = beam_emission_scores.device
        
        beam = beam if beam is not None else self.beam
        batch_size, seq_len = beam_emission_scores.size()[:2]

        traj_tokens, traj_scores = [], []
        traj_scores2 = []
        traj_tokens2 = []
        selected_probs = []
        finalized_tokens = []

        score = beam_emission_scores[:, 0]
        _score2 = beam_emission_scores[:,0][:,None,:].expand( -1, self.num_samples, -1 )
        _score3 = beam_emission_scores[:,0][:,None,:,None].expand( -1, self.num_samples, -1, beam )
        B, N, C, W = _score3.shape
        flat_score = _score3.permute(0, 3, 1, 2).reshape(-1, C)
        logits = flat_score / self.temp

        #torch.manual_seed(42)
        #_index_flat = torch.multinomial(probs, num_samples=1, replacement=True)
        batch_size, seq_len, N = sampled_beam_idx.size()
        sampled_beam_idx2 = sampled_beam_idx[:,0,:] # B, N
        sampled_beam_idx2 = sampled_beam_idx2.unsqueeze(1).repeat( 1, beam, 1 )
        sampled_beam_idx3 = sampled_beam_idx2.view( B * beam * N )
        # 行インデックスの配列 (0, 1, 2, ..., total-1) を作成
        _index_flat = sampled_beam_idx3.unsqueeze( -1 )
        
        all_inf_mask = (logits == float('-inf')).all(dim=-1)
        if all_inf_mask.any():
            # 全て-infの行だけ、一様な値をいれる
            logits[all_inf_mask] = 0.0 

        #log_probs_full = F.log_softmax(logits, dim=-1)
        #probs_full = torch.exp(log_probs_full)
        #selected_log_prob = torch.gather(log_probs_full, -1, _index_flat) # (B*W*N, 1)  1
        #selected_log_probs.append( selected_log_prob.view( B, W, self.num_samples ).transpose(1,2) )
        probs_full = F.softmax( logits, dim = -1 )
        selected_prob = torch.gather( probs_full, -1, _index_flat)
        selected_probs.append( selected_prob.view( B, W, self.num_samples) ) # S, B, W = beam, N
        
        for i in range(1, seq_len):
            traj_scores.append(score)
            traj_scores2.append( _score2 )
            _score2 = _score2[:,:,:,None] + beam_transition_matrix[:, i-1,None,:,:].expand( -1, self.num_samples,-1,-1) 
            
            B, N, C, W = _score2.shape
            flat_score = _score2.permute(0, 3, 1, 2).reshape(-1, C)
            
            logits = flat_score / self.temp
            
            #torch.manual_seed(42) 
            #_index_flat = torch.multinomial(probs, num_samples=1, replacement=True)
            # sampled_beam_idx B, S, N

            sampled_beam_idx2 = sampled_beam_idx[:,i-1,:] # B, N
            sampled_beam_idx2 = sampled_beam_idx2.unsqueeze(1).repeat( 1, beam, 1 )
            sampled_beam_idx3 = sampled_beam_idx2.view( B * beam * N )
            # 行インデックスの配列 (0, 1, 2, ..., total-1) を作成
            _index_flat = sampled_beam_idx3.unsqueeze( -1 )
            
            all_inf_mask = (logits == float('-inf')).all(dim=-1)
            if all_inf_mask.any():
                # 全て-infの行だけ、一様な値をいれる
                logits[all_inf_mask] = 0.0 

            #log_probs_full = F.log_softmax(logits, dim=-1)
            #probs_full = torch.exp(log_probs_full)
            #selected_log_prob = torch.gather(log_probs_full, -1, _index_flat) # (B*W*N, 1)  2
            #selected_log_probs.append( selected_log_prob.view( B, W, self.num_samples ).transpose(1,2) )
            probs_full = F.softmax( logits, dim = -1 ) 
            selected_prob = torch.gather( probs_full, -1, _index_flat)
            selected_probs.append( selected_prob.view( B, W, self.num_samples) )  # S, B, W = beam, N
           
            _score_flat = torch.gather(flat_score, -1, _index_flat)
            _index2 = _index_flat.view(B, W, self.num_samples).transpose(1,2)
            _score2 = _score_flat.view(B, W, self.num_samples).transpose(1,2)
            

            _score2 = _score2 + beam_emission_scores[:, i][:,None,:].expand(-1,self.num_samples,-1)

            score, index = _score2[:,0,:], _index2[:,0,:]
            traj_tokens.append(index)
            traj_tokens2.append( _index2 )

        #selected_log_probs = torch.stack( selected_log_probs, dim = 0 )
        #log_probs = selected_log_probs.permute( 1, 0, 3, 2 )
        ##beam_probs = torch.exp( log_probs )
        selected_probs = torch.stack( selected_probs, dim = 0 ) # S, B, beam, N
        beam_probs = selected_probs.transpose( 0, 1 ) # B, S, beam, N
        beam_probs = F.softmax( beam_probs, dim = 2 ) # B, S, beam, N  → B, S, N   一つ一つの B,S,N について、beam についての総和が1の確率
        log_probs = torch.log( beam_probs )

        # now running the back-tracing and find the best
        B, N, C = _score2.shape
        flat_score = _score2.permute(0, 1, 2).reshape(-1, C)

        sampled_beam_idx2 = sampled_beam_idx[:,-1,:] # B, N
        sampled_beam_idx3 = sampled_beam_idx2.view( B * N )
        _index_flat = sampled_beam_idx3.unsqueeze( -1 )        

        _score_flat = torch.gather(flat_score, -1, _index_flat)
        _index2 = _index_flat.view(B, self.num_samples, 1)
        _score2 = _score_flat.view(B, self.num_samples, 1)
        current_sampled_index = _index2[:, :] #(B, N, 1 )

        ## 3. 最初の要素として追加
        finalized_tokens.append(current_sampled_index) # (B, N, 1)
        #finalized_scores.append(probs.view( B, N, C ).gather(2, current_sampled_index)) # (B, N, 1)

        # 4. 過去に遡ってパスを復元
        # traj_tokens2 はループ内で append された各ステップのサンプリング結果
        for idx_step, scs_step in zip(reversed(traj_tokens2), reversed(traj_scores2)):
            # 直前のステップで選ばれたインデックスを使って、その前のインデックスを引く
            previous_pointer = finalized_tokens[-1]
            finalized_tokens.append(idx_step.gather(2, previous_pointer))
            #finalized_scores.append(scs_step.gather(2, previous_pointer))

        # 5. 逆順にして結合
        # 5. 逆順にして結合 (この時点では [seq_len, B, Group, 1] のリスト)
        finalized_tokens.reverse()

        sampled_beam_idx = torch.cat(finalized_tokens, 2) # [B, G, S]
        
        # 2. beam_targets [B, S, Beam] と合わせるために transpose
        # [B, G, S] -> [B, S, G] にして、各ステップ(S)ごとのビーム(G)を選択
        sampled_beam_idx = sampled_beam_idx.permute(0, 2, 1) # [B, S, G]
        
        return log_probs, sampled_beam_idx    

    '''
    def _viterbi_decode(self, emissions, masks=None, beam=None):
        beam = beam if beam is not None else self.beam
        batch_size, seq_len = emissions.size()[:2]
        beam_emission_scores, beam_targets = emissions.topk(beam, 2)
        beam_transition_score1 = self.E1(beam_targets[:, :-1])  # B x (T-1) x K x D
        beam_transition_score2 = self.E2(beam_targets[:, 1:])   # B x (T-1) x K x D
        beam_transition_matrix = torch.bmm(
            beam_transition_score1.view(-1, beam, self.rank),
            beam_transition_score2.view(-1, beam, self.rank).transpose(1, 2))
        beam_transition_matrix = beam_transition_matrix.view(batch_size, -1, beam, beam) # bsz, seq_len, beam, beam

        traj_tokens, traj_scores = [], []
        finalized_tokens, finalized_scores = [], []

        # compute the normalizer in the log-space
        score = beam_emission_scores[:, 0]  # B x K
        #print( "score size:", score.size() )
        dummy = torch.arange(beam, device=score.device).expand(*score.size()).contiguous()

        for i in range(1, seq_len):
            traj_scores.append(score)
            _score = score[:, :, None] + beam_transition_matrix[:, i-1] # bsz, beam, beam
            _score, _index = _score.max(dim=1) # bsz, beam     bsz, beam 
            _score = _score + beam_emission_scores[:, i] # bsz, beam

            if masks is not None:
                score = torch.where(masks[:, i: i+1], _score, score)
                index = torch.where(masks[:, i: i+1], _index, dummy)
            else:
                score, index = _score, _index
            traj_tokens.append(index)

        # now running the back-tracing and find the best
        best_score, best_index = score.max(dim=1)
        finalized_tokens.append(best_index[:, None])
        finalized_scores.append(best_score[:, None])

        for idx, scs in zip(reversed(traj_tokens), reversed(traj_scores)):
            previous_index = finalized_tokens[-1]
            finalized_tokens.append(idx.gather(1, previous_index))
            finalized_scores.append(scs.gather(1, previous_index))

        finalized_tokens.reverse()
        finalized_tokens = torch.cat(finalized_tokens, 1)
        finalized_tokens = beam_targets.gather(2, finalized_tokens[:, :, None])[:, :, 0]

        finalized_scores.reverse()
        finalized_scores = torch.cat(finalized_scores, 1)
        finalized_scores[:, 1:] = finalized_scores[:, 1:] - finalized_scores[:, :-1]

        return finalized_scores, finalized_tokens
    '''
    
    def _compute_many_values(self, emissions, targets, sampled_beam_idx = None, top_indices = None, grpo_mode = True, crf_mode = False, masks=None, beam=None):

        beam = beam if beam is not None else self.beam
        batch_size, seq_len = emissions.size()[:2]

        #if top_indices is not None:
        #    print( "DEBUG: 3 top_indices.dtype in many_values:", top_indices.dtype )
        
        if top_indices == None:
            beam_emission_scores, beam_targets = torch.topk( emissions, beam, -1)
        else:
            #print( "DEBUG: 4 top_indices if else:" )
            beam_emission_scores = torch.gather( emissions, -1, top_indices )
            beam_targets = top_indices
        
        beam_transition_score1 = self.E1(beam_targets[:, :-1])  # B x (T-1) x K x D
        beam_transition_score2 = self.E2(beam_targets[:, 1:])   # B x (T-1) x K x D
        beam_transition_matrix = torch.bmm(
            beam_transition_score1.view(-1, self.beam, self.rank),
            beam_transition_score2.view(-1, self.beam, self.rank).transpose(1, 2))
        beam_transition_matrix = beam_transition_matrix.view(batch_size, -1, beam, beam) # bsz, seq_len, beam, beam

        if not self.ref_t:
            traj_tokens, traj_scores = [], []
            #finalized_tokens, finalized_scores = [], []
            finalized_tokens = []

            # compute the normalizer in the log-space
            score = beam_emission_scores[:, 0]  # B x K
            dummy = torch.arange(beam, device=score.device).expand(*score.size()).contiguous()

            for i in range(1, seq_len):
                traj_scores.append(score)
                _score = score[:, :, None] + beam_transition_matrix[:, i-1] # bsz, beam, beam
                _score, _index = _score.max(dim=1) # bsz, beam     bsz, beam  step i-1 における 256 → 256 の max から 256 への遷移確率と 
                                                # 256 → 256 の前の 256 の max のインデックストークン
                                                # index b * 256 の 位置が i の token で、値が i-1 のtoken   

                _score = _score + beam_emission_scores[:, i] # bsz, beam i における 256 の遷移確率ではない確率を加える。i における 256 の全確率。

                if masks is not None:
                    score = torch.where(masks[:, i: i+1], _score, score)
                    index = torch.where(masks[:, i: i+1], _index, dummy)
                else:
                    score, index = _score, _index
                traj_tokens.append(index)

            #all_scores = traj_scores
            #all_scores.append( score )
            #all_scores = torch.stack( all_scores, dim = 0 ).transpose( 0, 1 )
            #top_probs = F.softmax( all_scores, dim = -1 )
            
            ## now running the back-tracing and find the best
            best_score, best_index = score.max(dim=1)
            finalized_tokens.append(best_index[:, None]) # 時刻 T における b*256 の確率最大の token
            #finalized_scores.append(best_score[:, None]) #時刻 T における b*256 の確率最大の score


            for idx, scs in zip(reversed(traj_tokens), reversed(traj_scores)): #idx,scs は、反転時刻 i と i-1における b * 256のトークンと確率
                previous_index = finalized_tokens[-1] # 時刻 Tなど 求めたいトークンと確率の一個後 における token　b * 1
                finalized_tokens.append(idx.gather(1, previous_index)) # 時刻 一個後iのトークン previou_index に至るための時刻i-1 のトークン
                                                                    # b* 256 の token から b * 1 の previous_idnex token で gather
                #finalized_scores.append(scs.gather(1, previous_index)) # 時刻一個後 i のトークンに至るための時刻 i-1 の確率

            finalized_tokens.reverse()
            finalized_tokens = torch.cat(finalized_tokens, 1)
            sampled_beam_idx = finalized_tokens
            finalized_tokens = beam_targets.gather(2, finalized_tokens[:, :, None])[:, :, 0]

            #finalized_scores.reverse()
            #finalized_scores = torch.cat(finalized_scores, 1)
            #finalized_scores[:, 1:] = finalized_scores[:, 1:] - finalized_scores[:, :-1]
        
      
        if not self.ref_t:
            if grpo_mode:
                if self.crf_coef != 0.0:
                    numerator = self._compute_score(emissions, targets)
                    denominator = self._compute_normalizer(emissions, targets)
                    crf_loss = - ( numerator - denominator ).mean() / seq_len
                else:
                    crf_loss = torch.zeros( (1), device = emissions.device, dtype = torch.float )
                log_probs, beam_probs, sampled_beam_idx, b_finalized_tokens,  = \
                    self._compute_grpo_samples( beam_emission_scores, beam_transition_matrix, beam_targets )
                return  finalized_tokens, log_probs, beam_probs, crf_loss, beam_targets, sampled_beam_idx, b_finalized_tokens
            else:
                if crf_mode:
                    if self.crf_coef != 0.0:
                        numerator = self._compute_score(emissions, targets)
                        denominator = self._compute_normalizer(emissions, targets)
                        crf_loss = - ( numerator - denominator ).mean() / seq_len
                    else:
                        crf_loss = torch.zeros( (1), device = emissions.device, dtype = torch.float )
                    return finalized_tokens, crf_loss
                else:
                    return finalized_tokens
        else:
            ref_sampled_log_probs, sampled_beam_idx = self._compute_ref_probs( \
                beam_emission_scores, beam_transition_matrix, sampled_beam_idx, top_indices )
            return ref_sampled_log_probs, sampled_beam_idx


### CaptioninTransformer モデルの定義

In [ ]:
class CaptioningTransformer(nn.Module):
    
    #img_size       :clip に入力する画像サイズ。
    #dim_embedding  :bert の隠れ次元
    #length_max     :固定長の seq_len。97。
    #vocab_size     :語彙数。
    #tokenizer      :tokenizer
    #dropout        :dropout の値
    #pad_token      :tokenizer.pad_token_id
    #use_repeat_logits_half: bert の出力の repeat している token の確率を小さくするか。 boolean
    #crf_coef       :crf_loss に乗ずる crf_coef。0 だと、crf_loss を計算しない。
    #temp           :_compute_grpo_samplesのmultinomial の前の softmax の温度。
    #top_p          :_compute_grpo_samples の top_p
    #top_k          :_compute_grpo_samples の top_k
    #num_samples    :_compute_grpo_samples の グループ数
    #ref_t          :model を refence_model とする場合 True。通常の場合 False
    
    def __init__(self, img_size: int,  dim_embedding: int, length_max: int, vocab_size: int, tokenizer, dropout: float = 0.0, \
                 pad_token_id: int=0, use_repeat_logits_half=False, crf_coef = 1.0, temp=0.5, top_p = 0.9, top_k = 50,
                 num_samples=16, crf_beam = 256, ref_t = False):
        super().__init__()

        #CLIP
        model_id = "openai/clip-vit-large-patch14-336"
        self.clip_model = CLIPVisionModel.from_pretrained(model_id )
        memory = self.clip_model( torch.randn( 1, 3, 336, 336 ) )
        memory = memory.last_hidden_state
        img_length = memory.size(1)
        clip_dim = memory.size(2)
        self.connector_pool = nn.AdaptiveAvgPool1d(length_max - 1 )
        self.connector_ln = nn.LayerNorm( clip_dim )
        self.connector_linear1 = nn.Linear( clip_dim, dim_embedding )
        self.connector_gleu = nn.GELU()
        self.connector_linear2 = nn.Linear( dim_embedding, dim_embedding )

       
        # Connector
        self.connector_pool = nn.AdaptiveAvgPool1d(length_max - 1 )
        # Down Sampling
        cls_token = memory[:, :1, :] # (bsz, 1, 1024)
        patch_tokens = memory[:, 1:, :] # (bsz, 576, 1024)
        # パッチ部分を 576 -> 96 に圧縮
        patch_tokens = patch_tokens.transpose(1, 2) # (bsz, 1024, 576)
        patch_tokens = self.connector_pool(patch_tokens)
        patch_tokens = patch_tokens.transpose(1, 2) # (bsz, 96, 1024)
        # CLSと結合して合計 97 トークンにする
        memory = torch.cat([cls_token, patch_tokens], dim=1) # (bsz, 97, 1024)

        self.pos_emb = PositionalEmbedding( dim_embedding )

        model_id = "google-bert/bert-large-uncased"
        self.bert = BertModel.from_pretrained( model_id )

        ## 単語出力分布計算
        self.ln_outputs = nn.LayerNorm( dim_embedding )
        self.linear = nn.Linear(dim_embedding, vocab_size)

        crf_low_rank = 32
        self.crf_beam_size = crf_beam
        top_dropout = 0.0
        tgt_padding_idx = tokenizer.pad_token_id
        print( "initialize crf_layer" )
        self.ref_t = ref_t
        #self.toplayer = TopLayer( vocab_size, dim_embedding, crf_low_rank, crf_beam, top_dropout, 
        #                          tgt_padding_idx, crf_coef = crf_coef, temp=temp, top_p = top_p, top_k = top_k,
        #                          num_samples=num_samples, ref_t = ref_t )
        self.crf_layer = DynamicCRF(num_embedding = vocab_size, low_rank = crf_low_rank, beam_size = crf_beam, 
                                    crf_coef=crf_coef, temp=temp, top_p = top_p, top_k = top_k, num_samples= num_samples, ref_t = ref_t )
        self.dim_embedding = dim_embedding
        self.use_repeat_logits_half = use_repeat_logits_half


    def mlp_connector(self, memory ):

        cls_token = memory[:, :1, :] # (bsz, 1, 1024)
        patch_tokens = memory[:, 1:, :] # (bsz, 576, 1024)

        # パッチ部分を 576 -> 96 に圧縮
        patch_tokens = patch_tokens.transpose(1, 2) # (bsz, 1024, 576)
        patch_tokens = self.connector_pool(patch_tokens)
        patch_tokens = patch_tokens.transpose(1, 2) # (bsz, 96, 1024)

        # CLSと結合して合計 97 トークンにする
        memory = torch.cat([cls_token, patch_tokens], dim=1) # (bsz, 97, 1024)

        memory = self.connector_ln( memory )
        memory = self.connector_linear1( memory )
        memory = self.connector_gleu( memory )
        memory = self.connector_linear2( memory )
        
        return memory

    def forward(self, images: torch.Tensor, targets: torch.Tensor, sampled_beam_idx = None, top_indices = None, grpo_mode = True, crf_mode = False ):

        self.device = images.device
        
        memory = self.clip_model( images ).last_hidden_state
        memory = self.mlp_connector( memory )
        memory += self.pos_emb( memory )
        
        outputs = self.bert( inputs_embeds = memory ).last_hidden_state
        outputs = self.ln_outputs( outputs )
        emissions = self.linear( outputs )
        
        if self.use_repeat_logits_half == True:
            emissions = repeat_logits_half( emissions )

        if not self.ref_t:
            if grpo_mode:
                #if top_indices is not None:
                #    #print( "DEBUG 4 top_indices.dtype at if grpo_mode:", top_indices.dtype )
                #finalized_tokens, log_probs, entropy, crf_loss, top_indices, sampled_beam_idx, b_finalized_tokens = \
                finalized_tokens, log_probs, beam_probs, crf_loss, top_indices, sampled_beam_idx, b_finalized_tokens = \
                    self.crf_layer._compute_many_values(emissions, targets, top_indices = top_indices )
                #return finalized_tokens, log_probs, entropy, crf_loss, emissions, top_indices, sampled_beam_idx, b_finalized_tokens
                return finalized_tokens, log_probs, beam_probs, crf_loss, emissions, top_indices, sampled_beam_idx, b_finalized_tokens
            else:
                if crf_mode:
                    finalized_tokens, crf_loss = self.crf_layer._compute_many_values( emissions, targets, grpo_mode = False, crf_mode = True )
                    return finalized_tokens, crf_loss, emissions
                else:
                    finalized_tokens = self.crf_layer._compute_many_values ( emissions, targets, grpo_mode = False ) 
                    return finalized_tokens
        else:
            ref_log_probs, sampled_beam_idx = \
                self.crf_layer._compute_many_values(emissions, targets, sampled_beam_idx, top_indices )
            return ref_log_probs, sampled_beam_idx
    
    def repeat_logits_half(self, emissions ):
        
        penalty = 1.2
        scores, preds = torch.max( emissions, 2 )
        masks = emissions == scores[:,:,None]
        masks = masks.permute( 1, 0, 2 )
        new_mask = torch.zeros( (  masks.size(1), masks.size(2)), device = emissions.device, dtype=torch.bool )
        new_masks = torch.zeros( ( masks.size(0), masks.size(1), masks.size(2)), device = emissions.device, dtype=torch.bool )
        for i, mask in enumerate( masks ):
            new_mask = torch.logical_or( mask,  new_mask  )
            new_masks[i] = new_mask
        new_masks = new_masks.transpose(0,1)
        first_true_mask = ( new_masks.int().cumsum(dim = 1 ) == 1 ) & new_masks
        new_masks = new_masks & ( ~first_true_mask )

        p_masks = emissions > 0
        m_masks = emissions < 0
        p_new_masks = p_masks & new_masks
        m_new_masks = p_masks & new_masks
        emissions2 = emissions.clone()
        emissions2[p_new_masks] = emissions[p_new_masks] / penalty
        emissions2[m_new_masks] = emissions2[m_new_masks] * penalty

        return emissions2


### 学習におけるハイパーパラメータやオプションの設定

In [ ]:
model_id = "google-bert/bert-large-uncased"
tokenizer = BertTokenizer.from_pretrained( model_id )
sos_token_id = tokenizer.encode( [ "[unused0]" ] )[1]
eos_token_id = tokenizer.encode( [ "[unused1]" ] )[1]

print( "sos_tokeni_d:", sos_token_id )
print( "eos_token_id:", eos_token_id )

class ConfigTrain(object):
    '''
    ハイパーパラメータ、システム共通変数の設定
    '''
    def __init__(self):

        # ハイパーパラメータ
        self.img_size = 336
        self.dim_embedding = 1024   # 埋め込み層の次元
        self.length_max = 97
        #self.lr_clip = 1.5e-7
        #self.lr_con = 7.5e-5
        #self.lr_bert = 1.5e-5            # 学習率
        #self.lr_others = 7.5e-5
        self.lr_clip = 2e-7 / 4
        self.lr_con = 1e-4 / 4
        self.lr_bert = 2e-5 / 4            # 学習率
        self.lr_others = 1e-4 / 4
        self.dropout = 0.0         # dropout確率
        #self.batch_size = 128       # ミニバッチ数
        self.batch_size = 40       # ミニバッチ数
        #self.batch_size = 32       # ミニバッチ数
        #self.batch_size = 16       # ミニバッチ数
        #self.batch_size = 8       # ミニバッチ数
        #self.batch_size = 4       # ミニバッチ数
        #self.batch_size = 1       # ミニバッチ数
        #self.num_epochs = 100       # エポック数→Colab無料版でテストする際は10未満に修正を推奨
        #self.num_epochs = 100       # エポック数→Colab無料版でテストする際は10未満に修正を推奨
        #self.num_epochs = 60       # エポック数→Colab無料版でテストする際は10未満に修正を推奨
        self.num_epochs = 5       # エポック数→Colab無料版でテストする際は10未満に修正を推奨
        self.use_amp = True
        #self.use_amp = False
        #self.use_saved_pth = True
        self.use_saved_pth = False
        self.model_id = "google-bert/bert-large-uncased"
        self.vocab_size = len( tokenizer )
        self.weight_decay = 0.1
        self.betas = (0.9, 0.999 )
        self.warmup = 0.1
        #self.alpha = 1.0
        self.crf_coef = 1.0
        self.ce_coef = 0.3
        self.use_repeat_logits_half = False
        self.window_size = 3
        self.temp = 0
        self.top_p = 1.0
        self.top_k = 0
        self.num_samples = 0
        self.beam = 256
        
        # パスの設定
        self.img_directory = '/mnt/ssd1/uchiyats/python_image_recognition-main/6_img_captioning/6_5_myoriginal_transformer_captioning/train2017'
        self.anno_file = '/mnt/ssd1/uchiyats/python_image_recognition-main/data/coco2014/captions_train2017.json'
        self.save_directory = './model'

        # 検証に使う学習セット内のデータの割合
        self.test_ratio = 0.1
        self.val_ratio = 0.1
        #self.val_ratio = 0.004
        #self.test_ratio = 0.004
        
        # 学習に使うデバイス
        #self.device = 'cuda'
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        #self.device = 'cpu'
        
        # データローダーに使うCPUプロセスの数
        #self.num_workers = 4
        self.num_workers = 0 if self.device == torch.device('cpu') else 12
        #self.num_workers = 0
        
        # 移動平均で計算する損失の値の数
        self.moving_avg = 100

### 学習を行う関数

In [ ]:
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True

#os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'true'
config = ConfigTrain()

model_id = "google-bert/bert-large-uncased"
tokenizer = BertTokenizer.from_pretrained(model_id)
pad_token_id = tokenizer.pad_token_id
cls_token_id = tokenizer.cls_token_id
sep_token_id = tokenizer.sep_token_id
sos_token_id = tokenizer.encode( [ "[unused0]" ] )[1]
eos_token_id = tokenizer.encode( [ "[unused1]" ] )[1]
a_token_id = tokenizer.encode( [ "a" ] )[1]
the_token_id = tokenizer.encode( [ "the" ] )[1]
and_token_id = tokenizer.encode( [ "and" ] )[1]
in_token_id = tokenizer.encode( [ "in" ] )[1]
we_token_id = tokenizer.encode( [ "we" ] )[1]
i_token_id = tokenizer.encode( [ "i" ] )[1]
he_token_id = tokenizer.encode( [ "he" ] )[1]
she_token_id = tokenizer.encode( [ "she" ] )[1]
it_token_id = tokenizer.encode( [ "it" ] )[1]
they_token_id = tokenizer.encode( [ "they" ] )[1]
period_token_id = tokenizer.encode( [ "." ] )[1]
comma_token_id = tokenizer.encode( [ "," ] )[1]
dbl_token_id = tokenizer.encode( [ '"' ] )[1]
sgl_token_id = tokenizer.encode( [ "'" ] )[1]

# 辞書サイズを保存
vocab_size = len( tokenizer )

# モデル出力用のディレクトリを作成
os.makedirs(config.save_directory, exist_ok=True)

# 画像のtransformsを定義
transforms = v2.Compose([
    v2.Resize((336, 336)),
    v2.AutoAugment(),
    #v2.ToTensor(),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    ## Coco データセット 2017 train の平均と標準偏差
    #v2.Normalize((0.456,0.427,0.401),(0.224,0.219,0.231) )
    # ImageNetデータセットの平均と標準偏差
    #v2.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
    # Clip Model の config から引用。
    v2.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711))
])

# COCOデータロードの定義
train_dataset = dataset.CocoCaptions(root=config.img_directory,
                        annFile=config.anno_file,
                        transform=transforms)

# Subset samplerの生成
#val_set, train_set = util.generate_subset(
#    train_dataset, config.val_ratio)
test_set, val_set, train_set = util.generate_subset_test_val_train(
    train_dataset, config.test_ratio, config.val_ratio )
    
# 学習時にランダムにサンプルするためのサンプラー
train_sampler = SubsetRandomSampler(train_set)

# DataLoaderを生成
collate_func_lambda = lambda x: util.collate_func(x, tokenizer, eos_token_id, pad_token_id, config.length_max )
train_loader = torch.utils.data.DataLoader(
                    train_dataset,
                    batch_size=config.batch_size,
                    num_workers=config.num_workers,
                    sampler=train_sampler,
                    pin_memory=True,
                    collate_fn=collate_func_lambda)
val_loader = torch.utils.data.DataLoader(
                    train_dataset,
                    batch_size=config.batch_size,
                    num_workers=config.num_workers,
                    sampler=val_set,
                    pin_memory=True,
                    collate_fn=collate_func_lambda)

test_loader = torch.utils.data.DataLoader(
                    train_dataset,
                    #batch_size=config.batch_size,
                    batch_size=1,
                    num_workers=config.num_workers,
                    sampler=test_set,
                    pin_memory=True,
                    collate_fn=collate_func_lambda)


print( "config.device:", config.device )
print( "学習セット数:",len( train_loader ) )
print( "評価セット数:",len( val_loader ))
print( "テストセット数:",len( test_loader ))
print( "use_amp:", config.use_amp )
print( "use_saved_pth:", config.use_saved_pth )

# モデルの定義
#model = CaptioningTransformer( config.img_size,
#    config.dim_embedding, config.length_max, config.vocab_size,
#    tokenizer, config.dropout, pad_token_id = tokenizer.pad_token_id,
#    use_repeat_logits_half = config.use_repeat_logits_half,
#    crf_coef = config.crf_coef )
#model.to(config.device)
model = CaptioningTransformer( config.img_size,
    config.dim_embedding, config.length_max, config.vocab_size,
    tokenizer, config.dropout, pad_token_id = tokenizer.pad_token_id,
    use_repeat_logits_half = config.use_repeat_logits_half,
    crf_coef = config.crf_coef, temp=config.temp, top_p = config.top_p, 
    top_k = config.top_k, num_samples=config.num_samples, crf_beam = config.beam )
model.to(config.device)
#crf_low_rank = 32
#crf_beam_size = 256
#top_dropout = 0.0
#tgt_padding_idx = tokenizer.pad_token_id
#toplayer = TopLayer( vocab_size, dim_embedding, crf_low_rank, crf_beam_size, top_dropout, tgt_padding_idx )

# 損失関数の定義
#criterion = nn.CrossEntropyLoss( ignore_index = tokenizer.pad_token_id, reduction = 'mean' )
#criterion = nn.CrossEntropyLoss( reduction = 'mean' )
log_softmax = nn.LogSoftmax( dim = 2 )
#softmax = nn.Softmax( dim = 2 )
criterion_nll = nn.NLLLoss( reduction = 'none' )
#criterion_nll = nn.NLLLoss( )
#criterion = nn.CrossEntropyLoss()

# 最適化手法の定義
# 最適化手法の定義
# Optimizerの生成, clipとそうでないモジュールとの
# パラメータで異なる学習率を適用
params_clip = []
params_bert = []
params_con = []
#params_cri = []
params_others = []
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        if 'clip_model' in name:
            params_clip.append(parameter)
        elif 'connector' in name:
            params_con.append(parameter)
        elif 'bert' in name and 'critical' not in name:
            params_bert.append(parameter)
        #elif 'critical' in name:
        #    params_cri.append(parameter)
        else:
            params_others.append(parameter)

param_groups = [
    {'params': params_clip, 'lr': config.lr_clip},
    {'params': params_con, 'lr': config.lr_con},
    {'params': params_bert, 'lr': config.lr_bert},
    #{'params': params_cri, 'lr': config.lr_cri},
    {'params': params_others, 'lr': config.lr_others}]



#optimizer = torch.optim.AdamW( model.parameters() , lr=config.lr)
#optimizer = torch.optim.AdamW( param_groups, weight_decay = config.weight_decay, betas=config.betas )
optimizer = torch.optim.AdamW( param_groups, weight_decay = config.weight_decay, betas=config.betas )

use_saved_pth = True
PATH = './model/model_ImgCap_SFT_COCO_curr.pth'

print( "use_saved_pth:", use_saved_pth )
print( "PATH:", PATH )
print( "exist saved_pth:", os.path.isfile(PATH) ) 
if use_saved_pth and os.path.isfile(PATH):
    checkpoint = torch.load(PATH, map_location=torch.device('cpu'))
    model.load_state_dict(checkpoint['model_state_dict'], strict = False)
    print( "model parameters were loaded")

    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    #scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    #device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    ## optimizerのstateを現在のdeviceに移す。これをしないと、保存前後でdeviceの不整合が起こる可能性がある。
    #for state in optimizer.state.values():
        #for k, v in state.items():
            #if isinstance(v, torch.Tensor):
                #state[k] = v.to(device)
    #begin_epoch = checkpoint['epoch']
    #loss = checkpoint['loss']
    #global_step = checkpoint['global_step']    
    begin_epoch = 0
    global_step = 0
else:
    begin_epoch = 0
    global_step = 0

#file_param = 1
print( "begin_epoch:", begin_epoch )
print( "global_step:", global_step )


param_groups = [
    {'params': params_clip, 'lr': config.lr_clip},
    {'params': params_con, 'lr': config.lr_con},
    {'params': params_bert, 'lr': config.lr_bert},
    #{'params': params_cri, 'lr': config.lr_cri},
    {'params': params_others, 'lr': config.lr_others}]


# 全ステップ数
num_global_steps = len( train_loader ) * config.num_epochs
print( "num_global_steps:", num_global_steps )
num_warmup_steps = num_global_steps * config.warmup
print( "num_warmup_steps:", num_warmup_steps )
#スケジューラーの定義
scheduler = get_linear_schedule_with_warmup( optimizer, num_warmup_steps, num_global_steps )    
#t_scheduler = get_linear_schedule_with_warmup( t_optimizer, num_warmup_steps, num_global_steps )    





PATH = "model/model_bert_large_NAR_PAD_curr.pth"
print( "use_saved_pth:", config.use_saved_pth )
print( "exist saved_pth:", os.path.isfile(PATH) ) 
use_saved_pth = config.use_saved_pth
if use_saved_pth and os.path.isfile(PATH):
    checkpoint = torch.load(PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    #device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    ## optimizerのstateを現在のdeviceに移す。これをしないと、保存前後でdeviceの不整合が起こる可能性がある。
    #for state in optimizer.state.values():
        #for k, v in state.items():
            #if isinstance(v, torch.Tensor):
                #state[k] = v.to(device)
    begin_epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    global_step = checkpoint['global_step']    
else:
    begin_epoch = 0
    global_step = 0

print( "begin_epoch:", begin_epoch )
print( "global_ste:", global_step )

len_tr_loader = len( train_loader )
train_param = len_tr_loader // 100
#train_param = len_tr_loader // 6
len_val_loader = len( val_loader )
#train_param = len_val_loader // 3
val_param = len_val_loader // 3
print( "train_param:", train_param )
print( "val_param:", val_param )

print( "epochs:", config.num_epochs )
print( "batch_size:", config.batch_size )
print( "lr_clip:", config.lr_clip )
print( "lr_con:", config.lr_con )
print( "lr_bert:", config.lr_bert )
print( "lr_others:", config.lr_others )
print( "weight_decay:", config.weight_decay )
print( "betas:", config.betas )
print( "crf_coef:", config.crf_coef )
print( "ce_coef:", config.ce_coef )
print( "use_repeat_logits_half:", config.use_repeat_logits_half )

# 学習経過の書き込み
now = datetime.datetime.now()
train_loss_file = '{}/MyOriginal_train_loss_{}.csv'\
    .format(config.save_directory, now.strftime('%Y%m%d_%H%M%S'))
with open(train_loss_file, 'a') as f:
    print(f'{len_tr_loader}', file=f) 
print( "train_loss_file:", train_loss_file )
val_loss_file = '{}/MyOriginal_val_loss_{}.csv'\
    .format(config.save_directory, now.strftime('%Y%m%d_%H%M%S'))
with open(val_loss_file, 'a') as f:
    print(f'{len_val_loader}', file=f) 
norm_file = '{}/norm_{}.csv'\
    .format(config.save_directory, now.strftime('%Y%m%d_%H%M%S'))

# 学習
val_loss_best = float('inf')

fn = bleu_score.SmoothingFunction().method7

# AMP用のスケーラー
scaler = GradScaler(enabled=config.use_amp)

pad_token_id = tokenizer.pad_token_id

for epoch in range(config.num_epochs):
    with tqdm(train_loader) as pbar:
    #with tqdm(val_loader) as pbar:
        pbar.set_description(f'[エポック {epoch + 1}]')

        # 学習モードに設定
        model.train()

        train_losses = deque()
        train_a = deque()
        train_b = deque()
        train_errors = deque()
        train_bleus = deque()
        for n_batch, (imgs, captions, caption_lengths) in enumerate( pbar ):
            # ミニバッチを設定
            imgs = imgs.to(config.device)
            captions = captions.to(config.device)
                
            optimizer.zero_grad()

            # 最後の単語から次を予測する必要はないため最後の単語を除外
            with autocast(str(config.device),enabled=config.use_amp):
                #logits = model( imgs )

                # 損失の計算
                # 単語軸が第1軸である必要があるため、転置
                #train_batch_crf_loss = model.toplayer( src_representation, src_input, tgt_input, is_training = True )
                #train_batch_crf_loss, topk_probs, topk_indicies = model.toplayer( src_representation, src_input, tgt_input, is_training = True )
                #top_logits, top_indices = torch.topk( logits, 256, dim = -1 )
                #train_batch_crf_loss, topk_probs, topk_indicies = model.toplayer( logits, top_logits, top_indices, \
                #                                                                  top_logits, captions, is_training = True )
                finalized_tokens, crf_loss, emissions = \
                    model( imgs, captions, top_indices = None, grpo_mode = False, crf_mode = True )
                a = config.crf_coef *crf_loss
                #log_probs = log_softmax( logits ).transpose(0,1)
                #train_nll_loss = calc_loss_nll( log_probs, captions )
                b = config.ce_coef * nn.CrossEntropyLoss()( emissions.transpose(1,2), captions )
                loss = a + b

            #hypo_ids = torch.argmax( logits, dim = 2 )
            hypo_ids = finalized_tokens
            
            scaler.scale(loss).backward()
            #scaler.unscale_(optimizer)
            #clip_grad_threshold = 1.0
            #torch.nn.utils.clip_grad_norm_(\
            #        model.parameters(),
            #        clip_grad_threshold)
            # オプティマイザにより，パラメータを更新する
            scaler.step(optimizer)
            scaler.update()            
            scheduler.step()

            #for name, param in model.named_parameters():
            #    print( name )
            
            norm0 = torch.sqrt( torch.norm( model.clip_model.vision_model.encoder.layers[0].self_attn.q_proj.weight.grad, p = 2 ) ).item()
            norm1 = torch.sqrt( torch.norm( model.bert.encoder.layer[23].attention.self.query.weight.grad, p = 2 ) ).item()
            norm_mean = torch.mean( torch.stack ([ torch.sqrt( torch.norm( param.grad, p = 2 ) ) \
                                                  for param in model.parameters() if param.grad is not None ] ) ).item()
            with open(norm_file, 'a') as f:
                print( "epcoch:", epoch, ", step:", global_step, ", norm0:", norm0, ", norm1:", norm1, ", norm_mean:", norm_mean, file=f  )
                f.flush()
            global_step += 1

            n = 0
            hypo_sentence = []
            ref_sentence = []
            hypo_sentence1 = []
            ref_sentence1 = []
            total_error = 0
            total_token_length = 0
            total_bleu = 0
            n2 = 0
            for (hypo_id, caption) in zip( hypo_ids, captions ):
                ##hypo = tokenizer.decode( hypo_id.tolist(), skip_special_tokens = True )
                #hypo = model.my_decode( hypo_id.tolist(), tokenizer )
                #hypo_tokens = tokenizer.tokenize( hypo )
                ##reference = tokenizer.decode( caption.tolist(), skip_special_tokens = True )
                #reference = model.my_decode( caption.tolist(), tokenizer )
                #ref_tokens = tokenizer.tokenize( reference )
                hypo = tokenizer.decode( [token_id for token_id in hypo_id.tolist() if token_id != pad_token_id] )
                reference = tokenizer.decode( [token_id for token_id in caption.tolist() if token_id != pad_token_id] )
                hypo_tokens = tokenizer.tokenize( hypo )
                ref_tokens = tokenizer.tokenize( reference )

                
                # 認識誤りを計算
                (error, substitute, 
                    delete, insert, ref_length) = \
                    levenshtein.calculate_error(hypo_tokens,
                                                    ref_tokens)
                
                # 誤り文字数を累積する
                total_error += error
                # 文字の総数を累積する
                total_token_length += ref_length

                bleu = bleu_score.sentence_bleu( [reference], hypo, smoothing_function=fn  )
        
                total_bleu += bleu                    
                    
                if n < 1 and n_batch == len( train_loader ) - 1 :
                    hypo_sentence.append( hypo )
                    ref_sentence.append( reference )
                if n < 1 and n_batch % train_param == 0:
                    hypo_sentence1.append( hypo )
                    ref_sentence1.append( reference )
                    
                n += 1
                n2 += 1
            
            avg_error = total_error / total_token_length * 100
            avg_bleu = total_bleu / n2 * 100
                
            # 学習時の損失をログに書き込み
            train_losses.append(loss.item())
            train_a.append(a.item())
            train_b.append(b.item())
            train_errors.append( avg_error )
            train_bleus.append( avg_bleu )
            #train_ciders.append( avg_cider )
            if len(train_losses) > config.moving_avg:
                train_losses.popleft()
                train_a.popleft()
                train_b.popleft()
                train_errors.popleft()
                train_bleus.popleft()
                #train_ciders.popleft()
            mean_loss = torch.Tensor(train_losses).mean().item()
            mean_a = torch.Tensor(train_a).mean().item()
            mean_b = torch.Tensor(train_b).mean().item()
            mean_error = torch.Tensor(train_errors).mean().item()
            mean_bleu = torch.Tensor(train_bleus).mean().item()
            pbar.set_postfix({
                'loss': mean_loss,
                'crf': mean_a,
                'ca': mean_b,
                'WER': mean_error,
                'BLEU': mean_bleu,
                #'CIDER': torch.Tensor(train_ciders).mean().item()
            })
            with open(train_loss_file, 'a') as f:
                print(f'{epoch}, {mean_loss}, {mean_a}, {mean_b}, {mean_error}, {mean_bleu}', file=f)
            print_flag = 1
            for ( hypo_se, ref_se ) in zip( hypo_sentence1, ref_sentence1 ):
                if print_flag == 1:
                    print( "lr clip     :", optimizer.param_groups[0]["lr"] )
                    print( "lr connector:", optimizer.param_groups[1]["lr"] )
                    print( "lr bert     :", optimizer.param_groups[2]["lr"] )
                    print( "lr others   :", optimizer.param_groups[3]["lr"] )
                    print_flag = 0
                print(f'Train epoch = {epoch}, loss = {mean_loss}, WER = {mean_error}, BLEU = {mean_bleu}')
                print( "refe:", ref_se )
                print( "hypo:", hypo_se )
                    
            for ( hypo_se, ref_se ) in zip( hypo_sentence, ref_sentence ):
                print(f'Train epoch = {epoch}, loss = {mean_loss}, WER = {mean_error}, BLEU = {mean_bleu}')
                print( "refe:", ref_se )
                print( "hypo:", hypo_se )
    # 学習率を表示
    #print(f'学習率: {optimizer.param_groups[0]['lr']}')
    train_loss = np.mean(train_losses)
    train_a1 = np.mean(train_a)
    train_b1 = np.mean(train_b)
    train_error = np.mean(train_errors )
    train_bleu = np.mean(train_bleus )
    print(f'Train loss: {train_loss}')
    print(f'Train crf: {train_a1}')
    print(f'Train ca: {train_b1}')
    print(f'Train WER: {train_error}')        
    print(f'Train BLEU: {train_bleu}')

    # 検証
    with tqdm(val_loader) as pbar:
        pbar.set_description(f'[検証]')

        # 評価モード
        model.eval()

        #val_losses = []
        #val_losses = deque()
        #val_a = deque()
        #val_b = deque()
        val_errors = deque()
        val_bleus = deque()
        for n_batch, (imgs, captions, caption_lengths) in enumerate( pbar ):

            # ミニバッチを設定
            imgs = imgs.to(config.device)
            captions = captions.to(config.device)
            #caption_lengths = torch.tensor( caption_lengths ).to(config.device)
                
            with torch.no_grad():
                #outputs = model( imgs )
                #finalized_scores, finalized_tokens, top_probs, top_indices, crf_loss, emissions = \
                #    model.inference( imgs, captions, top_indices = None )
                finalized_tokens, crf_loss, emissions = \
                    model( imgs, captions, top_indices = None, grpo_mode = False, crf_mode = True )

                #hypo_ids = torch.argmax( outputs, dim = 2 )
                hypo_ids = finalized_tokens
                ## 損失の計算
                ## 単語軸が第1軸である必要があるため、転置
                ##outputs = outputs.transpose(1, 2)
                #src_representation = outputs
                ##src_input = imgs
                #src_input = outputs
                #tgt_input = captions
                #train_batch_crf_loss = model.toplayer( src_representation, src_input, tgt_input, is_training = True )
                #log_probs = log_softmax( outputs.transpose(0,1) )
                ##bsz, seq = captions.size()
                ##loss_CA = - criterion_nll( log_prob.view( bsz * seq, -1 ) , captions.view( bsz * seq ) )
                #train_nll_loss = calc_loss_ca( log_probs, captions )
                #a = torch.mean( train_batch_crf_loss )
                #b = config.alpha * torch.mean( train_nll_loss )
                #loss = a + b
               
            n = 0
            hypo_sentence = []
            ref_sentence = []
            hypo_sentence1 = []
            ref_sentence1 = []
            total_error = 0
            total_token_length = 0
            total_bleu = 0
            n2 = 0
            for (hypo_id, caption) in zip( hypo_ids, captions ):
                ##hypo = tokenizer.decode( hypo_id.tolist(), skip_special_tokens = True )
                #hypo = model.my_decode( hypo_id.tolist(), tokenizer )
                #hypo_tokens = tokenizer.tokenize( hypo )
                ##reference = tokenizer.decode( caption.tolist(), skip_special_tokens = True )
                #reference = model.my_decode( caption.tolist(), tokenizer )
                #ref_tokens = tokenizer.tokenize( reference )
                hypo = tokenizer.decode( [token_id for token_id in hypo_id.tolist() if token_id != pad_token_id] )
                reference = tokenizer.decode( [token_id for token_id in caption.tolist() if token_id != pad_token_id] )
                hypo_tokens = tokenizer.tokenize( hypo )
                ref_tokens = tokenizer.tokenize( reference )
       
                # 認識誤りを計算
                (error, substitute, 
                    delete, insert, ref_length) = \
                    levenshtein.calculate_error(hypo_tokens,
                                                ref_tokens)
                    
                # 誤り文字数を累積する
                total_error += error
                # 文字の総数を累積する
                total_token_length += ref_length

                bleu = bleu_score.sentence_bleu( [reference], hypo, smoothing_function=fn  )
        
                total_bleu += bleu

                if n < 1 and n_batch == len( val_loader ) - 1:
                    hypo_sentence.append( hypo )
                    ref_sentence.append( reference )
                        
                if n < 1 and n_batch % val_param == 0:
                    hypo_sentence1.append( hypo )
                    ref_sentence1.append( reference )
                    
                n += 1
                n2 += 1
                
            avg_error = total_error / total_token_length * 100                    
            avg_bleu = total_bleu / n2 * 100

            # 学習時の損失をログに書き込み
            #val_losses.append(loss.item())
            #val_a.append(a.item())
            #val_b.append(b.item())
            val_errors.append( avg_error )
            val_bleus.append( avg_bleu )
            if len(val_errors) > config.moving_avg:
                #val_losses.popleft()
                #val_a.popleft()
                #val_b.popleft()
                val_errors.popleft()
                val_bleus.popleft()
            #mean_loss = torch.Tensor(val_losses).mean().item()
            #mean_a = torch.Tensor(val_a).mean().item()
            #mean_b = torch.Tensor(val_b).mean().item()
            mean_error = torch.Tensor(val_errors).mean().item()
            mean_bleu = torch.Tensor(val_bleus).mean().item()
            pbar.set_postfix({
                #'loss': mean_loss,
                #'crf': mean_a,
                #'ca': mean_b,
                'WER': mean_error,
                'BLEU': mean_bleu,
            })
            # Validation Lossをログに書き込み
            with open(val_loss_file, 'a') as f:
                print(f'{epoch}, {mean_error}, {mean_bleu}', file=f)
            
            for ( hypo_se, ref_se ) in zip( hypo_sentence1, ref_sentence1 ):
                print(f'Val epoch = {epoch}, WER = {mean_error}, BLEU = {mean_bleu}')
                print( "refe:", ref_se )
                print( "hypo:", hypo_se )
                    
            for ( hypo_se, ref_se ) in zip( hypo_sentence, ref_sentence ):
                print(f'Val epoch = {epoch}, WER = {mean_error}, BLEU = {mean_bleu}')
                print( "refe:", ref_se )
                print( "hypo:", hypo_se )
                    
    # Loss 表示
    #val_loss = np.mean(val_losses)
    val_error = np.mean( val_errors )
    val_bleu = np.mean( val_bleus )
    #print(f'Validation loss: {val_loss}')
    print(f'Validation WER: {val_error}')
    print(f'Validation BLEU: {val_bleu}')

    ## より良い検証結果が得られた場合、モデルを保存
            
    # モデルを保存
    torch.save({'epoch': epoch,
                'global_step': global_step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': loss,},
        f'{config.save_directory}/model_ImgCap_SFT_COCO_2_curr.pth')
    ## モデルを保存
        
# モデルを保存
torch.save({'epoch': epoch,
    'global_step': global_step,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(
    'scheduler_state_dict': scheduler.state_dict(),
    'loss': loss,},
    f'{config.save_directory}/model_ImgCap_SFT_COCO_2_final.pth')
